# Heston calibration study — chaos with a Legendre basis

Run and analyze 100 independent Heston calibrations with a two-driver Legendre chaos basis. Completed fits are checkpointed and reused when the study is run again.


## 1. Imports and data

Load project libraries and the calibration/test surfaces. Paths work from the repository or the notebook directory.

In [ ]:
from pathlib import Path
import importlib
import json
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Heston/Legendre"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.core import chaos as general_chaos_module
general_chaos_module = importlib.reload(general_chaos_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
from wiener_chaos.experiments import calibration as study
study = importlib.reload(study)
print(f"Study workflow loaded: {study.__file__}")

DATA_PATH = ROOT / "Heston/heston_data.pkl"
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Calibration settings

This study uses 10 Legendre functions per Brownian driver and chaos order 2. Heston projection uses the plain estimator. Calibration paths refresh every 50 steps, and the study saves each completed fit. The configuration below defines the current optimizer and Monte Carlo budgets.

The early-stopping loss gate is `1e-6`. Both regularization weights are zero,
including `anchor_lambda=0`, matching the calibration notebook.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 1  # Index of the GPU used for all model computations.
if torch.cuda.is_available() and CUDA_DEVICE_INDEX >= torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(1)

CONFIG = {
    # Number of independent fits in the study.
    "n_calibrations": 100,
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Time-basis family for the Brownian chaos expansion.
    "basis": "legendre",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 2,
    # Number of time-basis functions per driver.
    "M": 10,
    # Diagonal jitter that stabilizes basis covariance factorizations.
    "base_jitter": 1e-12,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 20_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 4,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 101,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 100,
        },
        "projection_control": "none",
        "projection_antithetic": False,
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["MC"] * 7,
        # MC paths per calibration maturity; None for quadrature.
        "N": [1_000_000, 500_000] + [100_000] * 5,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 7,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 7,
    },
    # Pricing allocations at held-out test maturities.
    "pricing_test": {
        # Integration method for each held-out maturity.
        "method": ["MC"] * 6,
        # MC paths per held-out maturity; None for quadrature.
        "N": [1_000_000, 1_000_000] + [100_000] * 4,
        # Independent MC batches per held-out maturity.
        "Ns": [1] * 6,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 6,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 10_000,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 30,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 1e-7,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 1e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": None,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
OUTPUT_BASE_DIR = STUDY_DIR / "Statistics calibration" / "study"
RUN_CALIBRATIONS = CONFIG["n_calibrations"]  # Maximum number of study fits to execute now.
BASELINE_DIR = None  # Optional separate study for comparison.
print(f"Device: {DEVICE} | This execution: {RUN_CALIBRATIONS} / {CONFIG['n_calibrations']} calibrations")
print(f"Results base directory: {OUTPUT_BASE_DIR}")
maturity_table = pd.DataFrame([
    {"set": label, "maturity": t, "pricing": method}
    for label, key in (("Calibration", "calibr"), ("Test", "test"))
    for t, method in zip(data[f"maturities_{key}"], CONFIG[f"pricing_{key}"]["method"])
]).sort_values("maturity", ignore_index=True)
display(maturity_table)

# Reuse completed studies automatically; set False to run or resume calibrations.
SAVED_RESULTS_DIR = OUTPUT_BASE_DIR
LOAD_SAVED_RESULTS = (study.resolve_study_directory(SAVED_RESULTS_DIR) / "results.json").is_file()


## 3. Study workflow

Run, checkpoint, and loading functions are in [calibration.py](../../wiener_chaos/experiments/calibration.py). Data, hyperparameters, and CUDA device are set here independently.


## 4. Run or load the study

Load the latest completed study automatically when available. Otherwise, run
or resume the configured calibrations. Set `LOAD_SAVED_RESULTS=False` explicitly
to run or resume calibrations. Completed runs are checkpointed, and a completed
study becomes the default for analysis and plotting automatically.


In [ ]:
if LOAD_SAVED_RESULTS:
    if SAVED_RESULTS_DIR is None:
        raise ValueError("Set SAVED_RESULTS_DIR to a completed study folder before loading.")
    records, OUTPUT_DIR, ANALYSIS_CONFIG = study.load_synthetic_results(SAVED_RESULTS_DIR)
    print(f"Loaded {len(records)} saved calibrations from {OUTPUT_DIR}")
else:
    with tqdm(total=RUN_CALIBRATIONS, desc="Heston Legendre calibrations", unit="calibration") as progress_bar:
        records, OUTPUT_DIR = study.run_study(
            data, CONFIG, OUTPUT_BASE_DIR, DATA_PATH,
            progress=study.tqdm_progress_callback(progress_bar),
        n_calibrations=RUN_CALIBRATIONS,
        )
    ANALYSIS_CONFIG = CONFIG
    print(f"Completed {len(records)} calibrations. Results saved in: {OUTPUT_DIR}")
runs = study.results_tables(records)


## 5. Fit quality, optimizer steps, and runtime

Calibration and test IV MAE are computed from fitted implied vols after each run and reported in volatility basis points. The optimizer monitors training price MSE in this configuration, so its best monitored value is labelled `best_training_price_mse`; it is not an IV error. Initialization time includes Heston fitting and Legendre projection; calibration time covers the chaos optimizer. A saved baseline is compared only when its input data and basis dimensions match this study. Starting coefficients are compared only when the initialization settings also match.


In [ ]:
run_table = runs.set_index("run").copy()
run_table["initialization_s"] = run_table["elapsed_seconds_init"]
run_table["calibration_s"] = run_table["elapsed_seconds_chaos_calibration"]
run_table["total_s"] = run_table["initialization_s"] + run_table["calibration_s"]
monitor_columns = [column for column in (
    "best_validation_iv_mae_bp", "best_training_price_mse", "best_validation_price_mse"
) if column in run_table]
quality_columns = ["iv_mae_calibration_bp", "iv_mae_test_bp", *monitor_columns,
                   "loss_ins", "loss_oos", "invalid_iv_calibration", "invalid_iv_test"]
fit_summary = run_table[quality_columns].agg(["count", "mean", "std", "min", "max"]).T
timing_summary = run_table[["initialization_s", "calibration_s", "total_s"]].agg(
    ["count", "mean", "std", "min", "max"]
).T
display(fit_summary)
display(timing_summary)
display(run_table[["calibration_iterations", "calibration_s",
                   "iv_mae_calibration_bp", "iv_mae_test_bp", *monitor_columns]])
if (run_table["calibration_iterations"] >= ANALYSIS_CONFIG["calibration"]["n_iter"]).any():
    print("WARNING: At least one run reached the iteration cap; check its IV errors before extending the study.")

baseline_rows = []
baseline_manifest_path = None if BASELINE_DIR is None else BASELINE_DIR / "manifest.json"
if baseline_manifest_path is None or not baseline_manifest_path.is_file():
    print("No baseline study selected." if baseline_manifest_path is None else
          f"Skipping baseline comparison: no manifest at {baseline_manifest_path}")
else:
    baseline_identity = json.loads(baseline_manifest_path.read_text())["identity"]
    current_identity = json.loads((OUTPUT_DIR / "manifest.json").read_text())["identity"]
    baseline_config = baseline_identity["config"]
    basis_keys = ("basis", "d", "M", "P")
    basis_matches = all(
        baseline_config.get(key) == ANALYSIS_CONFIG.get(key) for key in basis_keys
    )
    data_matches = baseline_identity.get("data_sha256") == current_identity.get("data_sha256")
    if not basis_matches or not data_matches:
        print("Skipping baseline comparison: saved runs use different input data or basis settings "
              f"(baseline M={baseline_config.get('M')}, current M={ANALYSIS_CONFIG.get('M')}).")
    else:
        same_initialization = (
            baseline_config.get("initialization") == ANALYSIS_CONFIG.get("initialization")
        )
        for record in records:
            baseline_path = BASELINE_DIR / "runs" / f"run_{record['run']:03d}" / "result.json"
            if not baseline_path.is_file():
                continue
            baseline = json.loads(baseline_path.read_text())
            current_initial = np.asarray(record["initial_coefficients"], dtype=float)
            baseline_initial = np.asarray(baseline["initial_coefficients"], dtype=float)
            if current_initial.shape != baseline_initial.shape:
                print(f"Skipping baseline run {record['run']}: coefficient shapes differ "
                      f"({current_initial.shape} versus {baseline_initial.shape}).")
                continue
            initial_difference = (
                float(np.max(np.abs(current_initial - baseline_initial)))
                if same_initialization else np.nan
            )
            baseline_rows.append({
                "run": record["run"],
                "initial_max_abs_difference": initial_difference,
                "old_calibration_s": baseline["elapsed_seconds_chaos_calibration"],
                "optimized_calibration_s": record["elapsed_seconds_chaos_calibration"],
                "old_calibration_iv_bp": baseline["iv_calibration_metrics"]["mae"] * 1e4,
                "optimized_calibration_iv_bp": record["iv_calibration_metrics"]["mae"] * 1e4,
                "old_test_iv_bp": baseline["iv_test_metrics"]["mae"] * 1e4,
                "optimized_test_iv_bp": record["iv_test_metrics"]["mae"] * 1e4,
                "optimized_iterations": record["calibration_iterations"],
            })
if baseline_rows:
    paired_comparison = pd.DataFrame(baseline_rows).set_index("run")
    paired_comparison["speedup"] = (
        paired_comparison["old_calibration_s"] / paired_comparison["optimized_calibration_s"]
    )
    display(paired_comparison)
    display(paired_comparison[["old_calibration_s", "optimized_calibration_s",
                               "old_calibration_iv_bp", "optimized_calibration_iv_bp",
                               "old_test_iv_bp", "optimized_test_iv_bp", "speedup"]].mean().to_frame("mean").T)
    study.save_artifacts(OUTPUT_DIR, tables={"paired_calibration_comparison": paired_comparison})
    if same_initialization:
        print(f"Maximum starting-coefficient difference versus saved runs: "
              f"{paired_comparison['initial_max_abs_difference'].max():.3e}")
    else:
        print("Starting-coefficient comparison skipped: initialization settings differ.")
elif baseline_manifest_path is not None and baseline_manifest_path.is_file() and basis_matches and data_matches:
    print("No matching saved baseline runs found for a paired comparison.")
print(f"Sum of initialization times: {run_table['initialization_s'].sum():.2f}s | "
      f"Sum of calibration times: {run_table['calibration_s'].sum():.2f}s | "
      f"Combined: {run_table['total_s'].sum():.2f}s")
print(f"Runs with invalid calibration IVs: {int((run_table['invalid_iv_calibration'] > 0).sum())}")

In [ ]:
figures = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, column in (("Calibration", "iv_mae_calibration_bp"), ("Test", "iv_mae_test_bp")):
    axes[0].plot(run_table.index, run_table[column], ".", label=label)
axes[0].set(xlabel="Calibration run", ylabel="IV MAE (bp)", title="Fit quality across runs")
axes[0].legend()
run_table[["initialization_s", "calibration_s", "total_s"]].plot(ax=axes[1])
axes[1].set(xlabel="Calibration run", ylabel="Seconds", title="Initialization and calibration time")
fig.tight_layout()
plt.show()
figures["study_summary"] = fig

## 6. Saved results

The study folder contains `manifest.json` and aggregate `results.json`, with
one checkpoint per calibration in `runs/run_XXX/result.json`. Coefficient tensors
are saved in `coefficients/`; fit and timing tables in `tables/`; summary plots
in `figures/`. Additional coefficient diagnostics are grouped in `diagnostics/`.
The relative `study_latest.json` pointer selects the completed study for loading
and plotting automatically.


In [ ]:
study.save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": torch.tensor([r["initial_coefficients"] for r in records], dtype=torch.float32),
                  "calibrated_coefficients": torch.tensor([r["d_a"] for r in records], dtype=torch.float32)},
    tables={"calibrations": run_table, "fit_summary": fit_summary, "timing_summary": timing_summary},
    figures=figures,
)
print(f"Study results saved in: {OUTPUT_DIR}")
